In [1]:
import os

import urllib.error
import urllib.request
from dotenv import load_dotenv

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

True

In [3]:
SYSTEM_PROMPT = """You are a literary data assistant.

## Capabilities

- `fetch_text_from_url`: loads document text from a URL into the conversation.
Do not guess line counts or positions—ground them in tool results from the saved file."""

In [4]:
content = f"""Project Gutenberg hosts a full plain-text copy of F. Scott Fitzgerald's The Great Gatsby.
URL: https://www.gutenberg.org/files/64317/64317-0.txt

Answer as much as you can:

1) How many lines in the complete Gutenberg file contain the substring `Gatsby` (count lines, not occurrences within a line, each line ends with a line break).
2) The 1-based line number of the first line in the file that contains `Daisy`.
3) A two-sentence neutral synopsis.

Do your best on (1) and (2). If at any point you realize you cannot **verify** an exact answer with
your available tools and reasoning, do not fabricate numbers: use `null` for that field and spell out
the limitation in `how_you_computed_counts`. If you encounter any errors please report what the error was and what the error message was."""

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Global storage for document chunks
document_chunks = {}
document_full_text = {}

@tool
def fetch_text_from_url(url: str) -> str:
    """
        Fetch the document from a URL and chunk it to stay within token limits.
        Returns info about chunks; use get_chunk() to retrieve specific chunks.
    """
    global document_chunks, document_full_text
    
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"
    text = raw.decode("utf-8", errors="replace")

    # Store full text for line-by-line analysis
    document_full_text[url] = text
    
    # Chunk the text to avoid token limit issues
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,  # Size of each chunk in characters
        chunk_overlap=200,  # Overlap between chunks for context
        separators=["\n\n", "\n", " ", ""]  # Split by paragraphs first
    )
    
    chunks = text_splitter.split_text(text)
    document_chunks[url] = chunks
    
    # Return comprehensive info
    chunk_info = f"Document fetched successfully from {url}\n"
    chunk_info += f"Total chunks: {len(chunks)}\n"
    chunk_info += f"Total lines in document: {len(text.splitlines())}\n\n"
    chunk_info += f"[CHUNK 1 of {len(chunks)}]\n{chunks[0]}\n\n"
    chunk_info += "Use get_chunk(url, chunk_number) to fetch other chunks.\n"
    chunk_info += "Use analyze_full_text(url) to search the entire document.\n"
    
    return chunk_info

@tool
def get_chunk(url: str, chunk_number: int) -> str:
    """Retrieve a specific chunk from a previously fetched document."""
    global document_chunks
    if url not in document_chunks:
        return f"Error: Document from {url} not found. Fetch it first using fetch_text_from_url."
    chunks = document_chunks[url]
    if chunk_number < 1 or chunk_number > len(chunks):
        return f"Error: Chunk {chunk_number} out of range. Document has {len(chunks)} chunks."
    return f"[CHUNK {chunk_number} of {len(chunks)}]\n{chunks[chunk_number - 1]}"

@tool
def analyze_full_text(url: str) -> str:
    """Analyze the full document text for line counts, searches, etc."""
    global document_full_text
    if url not in document_full_text:
        return f"Error: Document from {url} not loaded. Fetch it first."
    
    text = document_full_text[url]
    lines = text.splitlines()
    
    # Count lines with "Gatsby"
    gatsby_lines = [i+1 for i, line in enumerate(lines) if "Gatsby" in line]
    gatsby_count = len(gatsby_lines)
    
    # Find first line with "Daisy"
    daisy_line = None
    for i, line in enumerate(lines):
        if "Daisy" in line:
            daisy_line = i + 1
            break
    
    result = f"Full text analysis for {url}:\n"
    result += f"Total lines: {len(lines)}\n"
    result += f"Lines containing 'Gatsby': {gatsby_count}\n"
    result += f"First line with 'Daisy': {daisy_line}\n"
    return result

In [6]:
llm = ChatOpenAI(
    model='gpt-3.5-turbo',
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("API_KEY"),
    temperature=0.5,
    timeout=None,
    max_retries=2,
)

In [7]:
checkpointer = InMemorySaver()

In [8]:
agent = create_agent(
    model=llm,
    tools=[fetch_text_from_url, get_chunk, analyze_full_text],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer
)

In [9]:
agent_result = agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-lc"}},
)

In [10]:
print(agent_result["messages"][-1].content_blocks)

[{'type': 'text', 'text': "Here are the answers based on the analysis of the text from the Project Gutenberg file of *The Great Gatsby*:\n\n1) **Lines containing the substring `Gatsby`:** 258 lines contain the substring `Gatsby`.\n\n2) **First line number containing `Daisy`:** The first line that contains the substring `Daisy` is line number 181.\n\n3) **Neutral synopsis:** *The Great Gatsby* is a novel set in the 1920s that explores themes of wealth, love, and the American Dream through the life of Jay Gatsby, a mysterious millionaire. The story is narrated by Nick Carraway, who becomes entangled in Gatsby's pursuit of his former lover, Daisy Buchanan."}]


In [28]:
import json

# Get the final response message
final_message = agent_result["messages"][-1]

# Extract content blocks
content_blocks = final_message.content_blocks if hasattr(final_message, 'content_blocks') else final_message.content

print("\n")
print("╔" + "═" * 78 + "╗")
print("║" + " AGENT RESPONSE ".center(78) + "║")
print("╚" + "═" * 78 + "╝")

if isinstance(content_blocks, list):
    for i, block in enumerate(content_blocks, 1):
        if isinstance(block, dict) and block.get("type") == "text":
            # Extract and properly display the text
            text_content = block.get("text", "")
            print(text_content)
        else:
            print(str(block))
else:
    print(str(content_blocks))

print("\n" + "═" * 80)



╔══════════════════════════════════════════════════════════════════════════════╗
║                                AGENT RESPONSE                                ║
╚══════════════════════════════════════════════════════════════════════════════╝
Here is what I can provide based on the fetched document data and tool outputs:

- Total lines in the complete Gutenberg file: 6407 (as reported by the fetch_text_from_url tool).

Questions and results:
1) How many lines in the complete Gutenberg file contain the substring "Gatsby" (count lines, not occurrences within a line, each line ends with a line break).
- Result: null
- How I computed (limitation): I could not perform a full line-by-line scan within the current session to count every line that contains the substring "Gatsby" across all 6407 lines. The tool output confirms total lines but does not provide a per-line tally. I would need to run a full analyze/search across all lines to give an exact count.

2) The 1-based line number of the 

In [11]:
# Run the full-text analysis to get exact counts
url = "https://www.gutenberg.org/files/64317/64317-0.txt"

# Since analyze_full_text is a StructuredTool, invoke it properly
analysis_result = analyze_full_text.invoke({"url": url})

print("FULL TEXT ANALYSIS RESULTS:")
print("=" * 80)
print(analysis_result)
print("=" * 80)

# Also display as formatted output
lines = analysis_result.split('\n')
print("\nFORMATTED RESULTS:")
print("-" * 80)
for line in lines:
    if line.strip():
        print(f"  {line}")
print("-" * 80)

FULL TEXT ANALYSIS RESULTS:
Full text analysis for https://www.gutenberg.org/files/64317/64317-0.txt:
Total lines: 6407
Lines containing 'Gatsby': 258
First line with 'Daisy': 181


FORMATTED RESULTS:
--------------------------------------------------------------------------------
  Full text analysis for https://www.gutenberg.org/files/64317/64317-0.txt:
  Total lines: 6407
  Lines containing 'Gatsby': 258
  First line with 'Daisy': 181
--------------------------------------------------------------------------------
